# 🎥 ROTBOTS — Video Generator

---

Generates video clips from prompts via fal.ai.

> **🤔** What does it cost (financially, environmentally) to generate these clips?

---

In [ ]:
# SETUP
!pip install -q fal-client requests
import fal_client, json, os, time as _time, requests, threading
from pathlib import Path
from IPython.display import display, Markdown, Video, HTML

from google.colab import drive
drive.mount('/content/drive')
BASE_DIR = Path('/content/drive/MyDrive/rotbots_workshop')

def progress_bar(current, total, label='', width=30):
    pct = current / total if total > 0 else 0
    filled = int(width * pct)
    bar = '█' * filled + '░' * (width - filled)
    return f'<div style="font-family:monospace;font-size:14px;padding:2px 0;">{bar} {current}/{total} {label} ({pct:.0%})</div>'

print('✅ Setup complete')

In [ ]:
# API KEY
FAL_API_KEY = ''  # <-- fal.ai key here
if not FAL_API_KEY: print('⚠️  Paste fal.ai key above! https://fal.ai/dashboard/keys')
else: os.environ['FAL_KEY'] = FAL_API_KEY; print('✅ fal.ai configured')

In [ ]:
# SELECT SESSION
sessions = sorted([d.name for d in BASE_DIR.iterdir() if d.is_dir() and (d / 'session_info.json').exists()])
if not sessions: print('❌ No sessions!')
else:
    print('📂 Sessions:')
    for i,s in enumerate(sessions): print(f'   {i}: {s}')
SESSION_NAME = sessions[-1] if sessions else ''
SESSION_DIR = BASE_DIR / SESSION_NAME
VIDEOS_DIR = SESSION_DIR / 'videos'; VIDEOS_DIR.mkdir(exist_ok=True)
print(f'✅ Session: {SESSION_NAME}')

In [ ]:
# LOAD PROMPTS
pf = SESSION_DIR / 'prompts.json'
if pf.exists():
    with open(pf) as f: prompts = json.load(f)
    print(f'✅ {len(prompts)} prompts loaded')
    for p in prompts: print(f'   Scene {p["scene"]}: {p["title"]} [{p["style"]}] ({p["duration"]}s)')
else: print('❌ No prompts.json!')

In [ ]:
# MODEL
MODELS = {
    'wan-t2v': {'endpoint':'fal-ai/wan-t2v','desc':'Wan 2.1 (~$0.10/s)','ds':5},
    'wan-2.2': {'endpoint':'fal-ai/wan/v2.2-a14b/text-to-video','desc':'Wan 2.2 A14B ($0.10/s)','ds':5},
    'minimax': {'endpoint':'fal-ai/minimax/video-01','desc':'MiniMax (~$0.28/vid)','ds':6},
    'ltx-video': {'endpoint':'fal-ai/ltx-video/v2.1','desc':'LTX 2.1 (fast)','ds':5},
}
CHOSEN_MODEL = 'wan-t2v'
model = MODELS[CHOSEN_MODEL]
print(f'🎥 {CHOSEN_MODEL}: {model["desc"]}')

In [ ]:
# GENERATE with live timer
prog = display(HTML(''), display_id=True)
generated_videos = []
t0 = _time.time()
_stop_timer = False

def _run_timer(prog_handle, scene_info, done_count, total, t_start):
    """Background thread that updates the progress every second."""
    spinners = ['⠋','⠙','⠹','⠸','⠼','⠴','⠦','⠧','⠇','⠏']
    tick = 0
    while not _stop_timer:
        elapsed = _time.time() - t_start
        clip_elapsed = _time.time() - scene_info.get('clip_start', t_start)
        avg_per = elapsed / done_count[0] if done_count[0] > 0 else 0
        eta = (total - done_count[0] - 1) * avg_per + max(0, avg_per - clip_elapsed) if avg_per > 0 else 0
        s = spinners[tick % len(spinners)]
        pct = done_count[0] / total if total > 0 else 0
        filled = int(30 * pct)
        bar = '█' * filled + '░' * (30 - filled)
        html = (
            f'<div style="background:#1a1a2e;padding:12px;border-radius:8px;color:#eaeaea;font-family:monospace;">'
            f'<div style="font-size:16px;font-weight:bold;color:#e94560;">{s} Rendering Scene {scene_info.get("sn","?")}: {scene_info.get("title","")}</div>'
            f'<div style="font-size:14px;padding:2px 0;">{bar} {done_count[0]}/{total} done ({pct:.0%})</div>'
            f'<div style="color:#a0a0a0;font-size:12px;margin-top:4px;">'
            f'⏱️ {elapsed:.0f}s total · this clip: {clip_elapsed:.0f}s · ~{eta:.0f}s remaining</div>'
            f'<div style="color:#666;font-size:11px;margin-top:2px;">{scene_info.get("prompt","")[:70]}...</div>'
            f'</div>'
        )
        try: prog_handle.update(HTML(html))
        except: pass
        tick += 1
        _time.sleep(1)

for i, pd in enumerate(prompts):
    sn = pd['scene']; dur = pd.get('duration', model['ds'])
    scene_info = {'sn': sn, 'title': pd['title'], 'prompt': pd['t2v_prompt'], 'clip_start': _time.time()}
    done_count = [len(generated_videos)]
    
    # Start background timer
    _stop_timer = False
    timer = threading.Thread(target=_run_timer, args=(prog, scene_info, done_count, len(prompts), t0), daemon=True)
    timer.start()
    
    try:
        inp = {'prompt': pd['t2v_prompt']}
        if 'wan' in CHOSEN_MODEL: inp['aspect_ratio']='16:9'; inp['enable_prompt_expansion']=False
        elif CHOSEN_MODEL=='ltx-video': inp['aspect_ratio']='16:9'
        result = fal_client.subscribe(model['endpoint'], arguments=inp)
        url = None
        if isinstance(result, dict):
            if 'video' in result and isinstance(result['video'], dict): url = result['video'].get('url')
            elif 'video' in result: url = result['video']
            elif 'url' in result: url = result['url']
        if url:
            vp = VIDEOS_DIR / f'scene_{sn:03d}.mp4'
            with open(vp, 'wb') as f: f.write(requests.get(url, timeout=120).content)
            clip_time = _time.time() - scene_info['clip_start']
            generated_videos.append({'scene':sn,'title':pd['title'],'path':str(vp),'duration':dur,'source':'generated','gen_time':round(clip_time,1)})
    except Exception as e:
        _stop_timer = True; timer.join(timeout=2)
        prog.update(HTML(
            f'<div style="background:#1a1a2e;padding:12px;border-radius:8px;color:#eaeaea;font-family:monospace;">'
            f'<div style="font-size:16px;font-weight:bold;color:#e94560;">❌ Scene {sn} failed</div>'
            f'<div style="color:#a0a0a0;font-size:12px;">{str(e)[:80]}</div></div>'
        ))
        _time.sleep(3)
        continue
    finally:
        _stop_timer = True
        timer.join(timeout=2)

elapsed = _time.time() - t0
avg = elapsed / len(prompts) if prompts else 0
prog.update(HTML(
    f'<div style="background:#1a1a2e;padding:12px;border-radius:8px;color:#eaeaea;font-family:monospace;">'
    f'<div style="font-size:16px;font-weight:bold;color:#4ecca3;">✅ {len(generated_videos)}/{len(prompts)} videos generated</div>'
    f'<div style="font-size:14px;padding:2px 0;">'
    f'{"█" * 30} {len(generated_videos)}/{len(prompts)} done (100%)</div>'
    f'<div style="color:#a0a0a0;font-size:12px;margin-top:4px;">'
    f'Total: {elapsed:.0f}s · Avg: {avg:.0f}s/clip · Model: {CHOSEN_MODEL}</div></div>'
))

In [ ]:
# FOUND FOOTAGE (optional)
USE_FOUND_FOOTAGE = False
if USE_FOUND_FOOTAGE:
    from google.colab import files
    for fn,content in files.upload().items():
        dest=VIDEOS_DIR/fn
        with open(dest,'wb') as f: f.write(content)
        generated_videos.append({'scene':100+len(generated_videos),'title':f'Found: {fn}','path':str(dest),'duration':0,'source':'found_footage'})
else: print('ℹ️ Set USE_FOUND_FOOTAGE = True to upload clips')

In [ ]:
# PREVIEW
generated_videos.sort(key=lambda x:x['scene'])
for v in generated_videos:
    tag='🤖' if v['source']=='generated' else '📂'
    gt = f" ({v['gen_time']}s)" if v.get('gen_time') else ''
    display(Markdown(f'### {tag} Scene {v["scene"]}: {v["title"]}{gt}'))
    if Path(v['path']).exists():
        try: display(Video(v['path'],embed=True,width=640))
        except: print(f'   File: {v["path"]}')
    display(Markdown('---'))

In [ ]:
# SAVE MANIFEST
with open(SESSION_DIR/'video_manifest.json','w') as f: json.dump({'model':CHOSEN_MODEL,'videos':generated_videos},f,indent=2)
print(f'✅ {len(generated_videos)} clips saved to session "{SESSION_NAME}"')

---
*ROTBOTS Workshop — LI-MA 2026*